# Registering conditions onto the features DataFrame

This extends the features table produced by `example_workflow.ipynb`
(one row per well, one column per feature) with per-well experiment
metadata from a layout file — cell line, treatment, dose, batch,
differentiation date, etc.

Well labels (A1..D6) repeat on every plate, so the join key has to be
**(recording, well)**, not `well` alone — merging on `well` by itself
would cross-wire conditions between recordings as soon as more than one
file is involved.

Real layout files (e.g. `layout/Bioni37A_3_050325.csv`) are one row per
well with explicit metadata columns, not the row/column-letter grid that
`MEA_features.read_layout` expects — so here the layout is just read
with pandas and merged onto `df`.

In [6]:
import os
import pandas as pd

## 1) Load the features table from `example_workflow.ipynb`

Reuses that notebook's own output instead of recomputing features from
the raw recording.

In [7]:
df = pd.read_csv(os.path.join('results', 'features_table.csv'))
df.shape

(24, 26)

## 2) Load the layout

`recording_name` + `well` is the join key; `exclude` flags wells that
shouldn't be analyzed; `treatment`/`dose`/`dose_unit` describe the
condition.

In [8]:
layout_path = os.path.join('layout', 'Bioni37A_3_050325.csv')
layout = pd.read_csv(layout_path)
layout[['recording_name', 'well', 'cell_line', 'treatment', 'dose', 'dose_unit', 'batch', 'div', 'exclude']].head()

,recording_name,well,cell_line,treatment,dose,dose_unit,batch,div,exclude
0,CACNA1E_Bioni37A_29_3_baseline_20250305_20250403,A1,Bioni37A,baseline,NaN,NaN,1,28,No
1,CACNA1E_Bioni37A_29_3_baseline_20250305_20250403,A2,Bioni37A,baseline,NaN,NaN,1,28,No
2,CACNA1E_Bioni37A_29_3_baseline_20250305_20250403,A3,Bioni37A,baseline,NaN,NaN,1,28,No
3,CACNA1E_Bioni37A_29_3_baseline_20250305_20250403,A4,Bioni37A,baseline,NaN,NaN,1,28,No
4,CACNA1E_Bioni37A_29_3_baseline_20250305_20250403,A5,Bioni37A,baseline,NaN,NaN,1,28,No


## 3) Match each well to its recording

`df`'s `file_path` identifies which recording a row came from; derive
the same `recording_name` the layout uses from it, so each well can be
matched to *its own* recording's layout row rather than just any row
with the same well label.

In [9]:
df['recording_name'] = df['file_path'].apply(lambda p: os.path.basename(p).rsplit('_spikes.csv', 1)[0])
df[['file_path', 'recording_name', 'well']].head()

,file_path,recording_name,well
0,data/CACNA1E_Bioni37A_29_3_baseline_20250305_2...,CACNA1E_Bioni37A_29_3_baseline_20250305_20250403,A1
1,data/CACNA1E_Bioni37A_29_3_baseline_20250305_2...,CACNA1E_Bioni37A_29_3_baseline_20250305_20250403,A2
2,data/CACNA1E_Bioni37A_29_3_baseline_20250305_2...,CACNA1E_Bioni37A_29_3_baseline_20250305_20250403,A3
3,data/CACNA1E_Bioni37A_29_3_baseline_20250305_2...,CACNA1E_Bioni37A_29_3_baseline_20250305_20250403,A4
4,data/CACNA1E_Bioni37A_29_3_baseline_20250305_2...,CACNA1E_Bioni37A_29_3_baseline_20250305_20250403,A5


## 4) Merge conditions onto df and drop excluded wells

In [10]:
def make_condition(row):
    """Human-readable condition label, e.g. 'baseline' or 'LTX5 1uM'."""
    if pd.isna(row['dose']):
        return row['treatment']
    return f"{row['treatment']} {row['dose']:g}{row['dose_unit']}"

layout = layout.copy()
layout['condition'] = layout.apply(make_condition, axis=1)

condition_cols = ['recording_name', 'well', 'cell_line', 'condition', 'treatment', 'dose', 'dose_unit',
                  'batch', 'div', 'differentiation_date', 'recording_date', 'exclude']
df = df.merge(layout[condition_cols], on=['recording_name', 'well'], how='left')

# Every well should have matched a layout row - a left join leaves NaNs
# behind for any (recording, well) pair the layout doesn't cover.
unmatched = df['condition'].isna()
assert not unmatched.any(), (
    f"{unmatched.sum()} well(s) had no matching layout row:\n"
    f"{df.loc[unmatched, ['recording_name', 'well']]}"
)

# Layout-flagged exclusions shouldn't be analyzed
df = df[df['exclude'] != 'Yes'].drop(columns='exclude').reset_index(drop=True)

df[['recording_name', 'well', 'condition', 'cell_line', 'batch', 'div', 'ISI [s]', 'FR [Hz]']].head()

,recording_name,well,condition,cell_line,batch,div,ISI [s],FR [Hz]
0,CACNA1E_Bioni37A_29_3_baseline_20250305_20250403,A1,baseline,Bioni37A,1,28,0.331929,4.980823
1,CACNA1E_Bioni37A_29_3_baseline_20250305_20250403,A2,baseline,Bioni37A,1,28,0.184327,6.083098
2,CACNA1E_Bioni37A_29_3_baseline_20250305_20250403,A3,baseline,Bioni37A,1,28,0.288130,4.351575
3,CACNA1E_Bioni37A_29_3_baseline_20250305_20250403,A4,baseline,Bioni37A,1,28,0.124922,8.505921
4,CACNA1E_Bioni37A_29_3_baseline_20250305_20250403,A5,baseline,Bioni37A,1,28,1.290092,6.676835


In [11]:
df

,file_path,well,ISI [s],CV_ISI,FR [Hz],IBI (MCS) [s],CV_IBI (MCS),Burst_Rate (MCS) [Hz],BD (MCS) [s],CV_BD (MCS),...,recording_name,cell_line,condition,treatment,dose,dose_unit,batch,div,differentiation_date,recording_date
0,data/CACNA1E_Bioni37A_29_3_baseline_20250305_2...,A1,0.331929,3.934419,4.980823,12.757490,1.023169,0.081542,0.401073,0.701178,...,CACNA1E_Bioni37A_29_3_baseline_20250305_20250403,Bioni37A,baseline,baseline,NaN,NaN,1,28,2025-03-05,2025-04-03
1,data/CACNA1E_Bioni37A_29_3_baseline_20250305_2...,A2,0.184327,4.596032,6.083098,11.305736,0.835156,0.088612,0.394648,0.588980,...,CACNA1E_Bioni37A_29_3_baseline_20250305_20250403,Bioni37A,baseline,baseline,NaN,NaN,1,28,2025-03-05,2025-04-03
2,data/CACNA1E_Bioni37A_29_3_baseline_20250305_2...,A3,0.288130,4.185486,4.351575,11.972195,0.823609,0.080842,0.323009,0.527778,...,CACNA1E_Bioni37A_29_3_baseline_20250305_20250403,Bioni37A,baseline,baseline,NaN,NaN,1,28,2025-03-05,2025-04-03
3,data/CACNA1E_Bioni37A_29_3_baseline_20250305_2...,A4,0.124922,3.865405,8.505921,10.576903,0.849551,0.093945,0.531548,0.640930,...,CACNA1E_Bioni37A_29_3_baseline_20250305_20250403,Bioni37A,baseline,baseline,NaN,NaN,1,28,2025-03-05,2025-04-03
4,data/CACNA1E_Bioni37A_29_3_baseline_20250305_2...,A5,1.290092,4.692344,6.676835,9.715796,0.878261,0.105176,0.438610,0.635979,...,CACNA1E_Bioni37A_29_3_baseline_20250305_20250403,Bioni37A,baseline,baseline,NaN,NaN,1,28,2025-03-05,2025-04-03
5,data/CACNA1E_Bioni37A_29_3_baseline_20250305_2...,A6,21.088807,0.988931,0.054752,NaN,NaN,NaN,NaN,NaN,...,CACNA1E_Bioni37A_29_3_baseline_20250305_20250403,Bioni37A,baseline,baseline,NaN,NaN,1,28,2025-03-05,2025-04-03
6,data/CACNA1E_Bioni37A_29_3_baseline_20250305_2...,B1,0.152231,4.474321,8.029799,9.593500,0.886055,0.108372,0.471836,0.606748,...,CACNA1E_Bioni37A_29_3_baseline_20250305_20250403,Bioni37A,baseline,baseline,NaN,NaN,1,28,2025-03-05,2025-04-03
7,data/CACNA1E_Bioni37A_29_3_baseline_20250305_2...,B2,0.286547,5.616671,5.393675,10.014617,1.342226,0.100304,0.480987,0.660319,...,CACNA1E_Bioni37A_29_3_baseline_20250305_20250403,Bioni37A,baseline,baseline,NaN,NaN,1,28,2025-03-05,2025-04-03
8,data/CACNA1E_Bioni37A_29_3_baseline_20250305_2...,B3,0.190194,4.973263,5.951359,12.089930,1.286200,0.087279,0.479422,0.756794,...,CACNA1E_Bioni37A_29_3_baseline_20250305_20250403,Bioni37A,baseline,baseline,NaN,NaN,1,28,2025-03-05,2025-04-03
9,data/CACNA1E_Bioni37A_29_3_baseline_20250305_2...,B4,0.213245,5.151095,6.240053,10.001109,1.066807,0.103357,0.375290,0.681565,...,CACNA1E_Bioni37A_29_3_baseline_20250305_20250403,Bioni37A,baseline,baseline,NaN,NaN,1,28,2025-03-05,2025-04-03


For many recording files at once, build `features_table.csv` across
all of them first (one row per well per recording, each with its own
`file_path`/`recording_name`), then do this same `(recording_name, well)`
merge once against the concatenated layout rows for every file. The
per-file processors in `MEA_features.py` (`process_recording_file`,
`process_multidrug_recording_file`, `process_plate_layout_recording_file`,
...) do this same per-file layout merge internally — see their
docstrings for which layout convention each one expects.